In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp4aljzht1".


In [2]:
%%cuda
#include <iostream>

__global__ void columnSum(float* matrix, float* sums, int width, int height) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < width && row < height) {
        float val = matrix[row * width + col];

        // Multiple threads in the same column will call this.
        // atomicAdd handles the "locking" so no data is lost.
        atomicAdd(&sums[col], val);
    }
}

int main() {
    int width = 4, height = 3;
    size_t matrixSize = width * height * sizeof(float);
    size_t sumSize = width * sizeof(float);

    // Matrix: 3 rows, 4 columns (all 1.0f)
    float h_matrix[] = {1, 1, 1, 1,
                        1, 1, 1, 1,
                        1, 1, 1, 1};
    float h_sums[4] = {0, 0, 0, 0};

    float *d_matrix, *d_sums;
    cudaMalloc(&d_matrix, matrixSize);
    cudaMalloc(&d_sums, sumSize);

    cudaMemcpy(d_matrix, h_matrix, matrixSize, cudaMemcpyHostToDevice);
    cudaMemset(d_sums, 0, sumSize); // Initialize GPU sums to 0

    dim3 threads(2, 2);
    dim3 blocks(2, 2);
    columnSum<<<blocks, threads>>>(d_matrix, d_sums, width, height);

    cudaMemcpy(h_sums, d_sums, sumSize, cudaMemcpyDeviceToHost);

    std::cout << "Column Sums: ";
    for(int i=0; i<width; i++) std::cout << h_sums[i] << " ";
    std::cout << "\n(Expected: 3.0 3.0 3.0 3.0)" << std::endl;

    cudaFree(d_matrix); cudaFree(d_sums);
    return 0;
}

Column Sums: 3 3 3 3 
(Expected: 3.0 3.0 3.0 3.0)

